# DiffusionGemma — EN→RU Translation
Zero-shot translation on WMT14 newstest2014 using Google's DiffusionGemma 26B.

**Before running:**
- Accelerator: GPU T4 x2 (or any GPU ≥ 16GB)
- Internet: ON
- Secrets: add `HF_TOKEN` (from huggingface.co → Settings → Access Tokens)

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers accelerate bitsandbytes sacrebleu tqdm pillow torchvision

In [ ]:
# Cell 2 — Auth + GPU check
import os, torch
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

print(f"CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3 — Download WMT14 EN→RU test set via sacrebleu
# Same 3003 sentences used in SeqDiffuSeq sessions — BLEU is directly comparable
import sacrebleu, shutil
from pathlib import Path

data_dir = Path("/kaggle/working/data/en-ru")
data_dir.mkdir(parents=True, exist_ok=True)

src_path = sacrebleu.get_source_file("wmt14", "en-ru")
ref_path = sacrebleu.get_reference_files("wmt14", "en-ru")[0]
shutil.copy(src_path, data_dir / "test.en")
shutil.copy(ref_path, data_dir / "test.ru")

src_lines = (data_dir / "test.en").read_text().splitlines()
ref_lines = (data_dir / "test.ru").read_text().splitlines()
print(f"Test set: {len(src_lines)} lines")
print(f"Example: {src_lines[0]}")

In [ ]:
# Cell 4 — Load DiffusionGemma with 4-bit quantization
# device_map='auto' splits across both T4s if 2xT4 is selected (32GB total)
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "google/diffusiongemma-26B-A4B-it"
HF_TOKEN = os.environ["HF_TOKEN"]

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

print("Loading model (4-bit)... this will take a few minutes on first run")
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    token=HF_TOKEN,
)
print("Model loaded!")
print(model.hf_device_map)

In [ ]:
# Cell 5 — Translation helper + smoke test (10 lines)
def translate(text, src_lang="English", tgt_lang="Russian", max_new_tokens=200):
    msgs = [{"role": "user", "content": [
        {"type": "text", "text": f"Translate the following {src_lang} text to {tgt_lang}:\n{text}"}
    ]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return processor.decode(
        out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
    ).strip()

# Smoke test — verify output looks like real Russian before full run
print("=== Smoke test (10 lines) ===")
for src in src_lines[:10]:
    hyp = translate(src)
    print(f"SRC: {src}")
    print(f"HYP: {hyp}")
    print()

In [ ]:
# Cell 6 — Full evaluation (3003 lines)
# Run only after smoke test confirms good output
from tqdm.notebook import tqdm

all_hyps = [translate(line) for line in tqdm(src_lines, desc="Translating")]

out_dir = Path("/kaggle/working/results/en-ru")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "hyps.ru").write_text("\n".join(all_hyps))

bleu = sacrebleu.corpus_bleu(all_hyps, [ref_lines], tokenize="13a")
report = (
    f"SacreBLEU (13a): {bleu.score:.2f}\n"
    f"Model: {MODEL_ID}\n"
    f"Lines: {len(all_hyps)}\n"
)
(out_dir / "bleu_report.txt").write_text(report)
print(report)